# 33장 · LangGraph 입문 · 상태와 노드로 워크플로 만들기

**LangGraph는 여러 처리 단계를 그래프로 정의하고, 입력 한 건이 그 단계를 따라 실행되게 하는 파이썬 라이브러리입니다.** 워크플로는 어떤 일을 어떤 순서와 조건으로 처리할지 정한 절차입니다. 앞 장에서는 파이썬 함수 안에서 다음 함수를 직접 불렀습니다. LangGraph에서는 각 함수가 맡을 일과 그 함수들을 실행할 순서를 따로 적습니다.

가령 문장이 열 글자 안에 드는지 판정하려면 글자 수를 세고, 그 수를 보고 판정 문구를 정합니다. 나중에 긴 문장만 고쳐서 다시 검사하게 하려면 ‘검사로 돌아가는 길’이 하나 더 필요합니다. LangGraph를 쓰면 어떤 단계가 값을 고쳤고 그다음 어디로 가는지 코드와 그림에서 따라갈 수 있습니다. 이번 장에서는 순서가 정해진 그래프를 만들고, 다음 장에서 조건에 따른 갈림길을 붙입니다. 이번 장의 모든 노드는 일반 파이썬 함수이므로 모델이나 API 키 없이 실행할 수 있습니다.

## 상태·노드·간선은 각각 무엇일까요?

숫자 2에 1을 더하는 그래프를 만들겠습니다. 입력은 `{"value": 2}`이고 기대하는 최종 상태는 `{"value": 3}`입니다. 이 작은 예에서 세 용어를 구분해 보세요.

| 용어 | 코드에서 가리키는 것 | 이 예에서의 역할 |
|---|---|---|
| 상태(state) | 한 번의 실행에서 단계들이 읽고 갱신하는 데이터 | `value`에 현재 숫자를 보관합니다. |
| 노드(node) | 그래프에 이름을 붙여 등록한 작업 | `plus_one`이라는 이름으로 등록할 함수가 현재 숫자를 읽고 새 값을 돌려줍니다. |
| 간선(edge) | 어느 작업 다음에 어디로 갈지 정한 연결 | 시작점에서 `plus_one`으로, 계산이 끝나면 종료점으로 이동합니다. |

그림의 상자는 작업을, 화살표는 다음에 실행할 방향을 나타냅니다. `START`와 `END`는 LangGraph가 제공하는 시작점과 종료점입니다. 두 지점에 실행할 함수를 따로 만들 필요는 없습니다.

새 커널이라면 코드 셀에서 `%pip install "langgraph>=1,<2"`를 한 번 실행하고 커널을 다시 시작하세요.

## StateGraph로 노드 하나 연결하기

`StateGraph(Count)`는 `Count`에 적어 둔 상태 필드를 사용할 그래프를 준비합니다. 아래의 `class Count(TypedDict)`는 `value`라는 키에 정수를 담겠다는 타입 선언입니다. 실행할 때 넘기는 값은 `{"value": 2}`처럼 평소에 쓰던 딕셔너리입니다. 먼저 상태를 받고 변경할 필드를 돌려주는 함수부터 작성합니다. `increment`에 딕셔너리를 직접 넣어서 그래프에 등록할 작업이 정확한지 확인하세요.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Count(TypedDict):
    value: int

def increment(state):
    return {"value": state["value"] + 1}

print(increment({"value": 2}))

{'value': 3}


함수의 반환값은 `{"value": 3}`입니다. 이제 같은 함수를 그래프에서 실행해 봅니다. 그림에서 가운데 상자의 이름이 코드의 `"plus_one"`입니다. `add_node("plus_one", increment)`의 둘째 인자에는 실행할 함수 자체를 넘깁니다. 두 화살표는 아래의 두 `add_edge` 줄과 하나씩 대응합니다.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    plus_one["plus_one · value에 1 더하기"]
    finish["END · 실행 마치기"]
    start --> plus_one
    plus_one --> finish
```

In [ ]:
builder = StateGraph(Count)
builder.add_node("plus_one", increment)
builder.add_edge(START, "plus_one")
builder.add_edge("plus_one", END)

graph = builder.compile()
print(graph.invoke({"value": 2}))

{'value': 3}


`compile()`은 등록한 노드와 연결을 확인해 실행할 수 있는 그래프를 만듭니다. 계산을 시작하는 줄은 `invoke({"value": 2})`입니다. 이때 그래프가 `increment`를 호출하고, 함수가 돌려준 3으로 상태의 `value`를 갱신합니다. END에 도착하면 `invoke`가 최종 상태를 돌려주므로 화면에 `{'value': 3}`이 보입니다.

입력의 2를 7로 바꿔 같은 그래프를 한 번 더 호출해 보세요. 새 실행은 7로 시작하고 8로 끝납니다. 연결 순서가 같으므로 그래프를 다시 조립할 필요가 없습니다.

## 두 노드가 하나의 상태를 이어 쓰는 방법

이번에는 문장을 넣으면 글자 수와 표시 문구가 함께 담긴 딕셔너리를 받도록 만듭니다. `count_chars`가 `count`를 기록하고, `decide_label`이 그 값을 읽어 `label`을 기록합니다. 각 함수를 먼저 호출해 본 뒤 두 노드를 연결합니다. [실습 수첩](pre-mid/documents/graph/workshop.md)에는 짝과 다른 입력을 시험할 때 쓸 기록 양식이 있습니다.

## TypedDict로 상태의 필드 정하기

`NoticeState`에 문장(`text`), 글자 수(`count`), 표시 문구(`label`)의 이름과 자료형을 적습니다. `TypedDict`는 코드에서 쓸 키를 명시하는 파이썬 타입 선언입니다. `total=False`는 일부 키가 없는 딕셔너리도 이 타입으로 표현할 수 있다는 뜻입니다. 처음에 `text`만 주면 나머지 두 키는 값을 계산한 뒤에 생깁니다.

아래 셀의 두 출력에서 함수가 돌려준 딕셔너리와 함수에 건넨 딕셔너리를 비교해 보세요.

In [1]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class NoticeState(TypedDict, total=False):
    text: str
    count: int
    label: str

def count_chars(state):
    # 원문은 그대로 두고 새로 셈한 칸만 돌려줍니다.
    return {"count": len(state["text"])}

sample = {"text": "창문을 닫아 주세요"}
print(count_chars(sample))
print(sample)

{'count': 10}
{'text': '창문을 닫아 주세요'}


함수를 직접 호출한 결과에는 count만 있고 원래 딕셔너리에는 text만 있습니다. 그래프가 노드 반환값을 상태에 반영하는 단계는 아직 실행하지 않았습니다. 공백도 한 글자로 셌습니다.

## 다음 노드가 읽을 상태 만들기

`decide_label`은 `count`가 10 이하면 “표시 가능”, 더 크면 “줄여 주세요”를 돌려줍니다. `{**sample, **count_chars(sample)}`은 원래 상태와 첫 함수의 반환값을 한 딕셔너리로 합칩니다. 여기서는 두 번째 함수가 받을 값을 직접 만들고, 다음 예제에서는 LangGraph가 같은 갱신을 수행합니다.

In [2]:
def decide_label(state):
    return {"label": "표시 가능" if state["count"] <= 10 else "줄여 주세요"}

# 첫 노드가 돌려준 칸을 더하면 둘째 노드의 입력이 됩니다.
after_count = {**sample, **count_chars(sample)}
print(after_count)
print(decide_label(after_count))

{'text': '창문을 닫아 주세요', 'count': 10}
{'label': '표시 가능'}


text와 count가 함께 있어도 두 번째 함수는 count만 읽습니다. 각 함수의 반환에 text를 계속 복사할 필요가 없습니다. 빈 문자열도 길이 0으로 통과하므로 ‘길이 통과’는 내용이 유효하다는 판단과는 구별해야 합니다.

## add_node와 add_edge로 두 노드 연결하기

그림의 두 작업은 방금 정의한 함수입니다. `count_chars`에서 `decide_label`로 이어지는 화살표를 코드에서 찾아보세요. 이 순서 덕분에 둘째 함수가 실행될 때는 이미 상태에 `count`가 있습니다. `build_notice()`는 실행 가능한 그래프를 만들고, `notice_graph.invoke(sample)`이 문장 하나를 그 그래프에 넣습니다.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    count_chars["count_chars · 글자 수 세기"]
    decide_label["decide_label · 길이로 표시 문구 정하기"]
    finish["END · 실행 마치기"]
    start --> count_chars
    count_chars --> decide_label
    decide_label --> finish
```

In [3]:
def build_notice():
    plan = StateGraph(NoticeState)
    plan.add_node("count_chars", count_chars)
    plan.add_node("decide_label", decide_label)
    plan.add_edge(START, "count_chars")
    plan.add_edge("count_chars", "decide_label")
    plan.add_edge("decide_label", END)
    return plan.compile()

notice_graph = build_notice()
print(notice_graph.invoke(sample))

{'text': '창문을 닫아 주세요', 'count': 10, 'label': '표시 가능'}


`count_chars`가 돌려준 딕셔너리에는 `count`만 있었지만 마지막 출력에는 원래의 `text`도 남아 있습니다. <mark>이 예에서 노드가 돌려주는 것은 상태에서 바꿀 필드입니다.</mark> 그래프는 먼저 글자 수를 반영하고, 다음 노드가 돌려준 `label`도 같은 상태에 반영합니다. 화면에 나온 최종 상태에서 세 키를 찾아보세요.

## stream으로 각 노드의 갱신 값 읽기

`invoke`로는 마지막 상태를 받았습니다. 어느 노드에서 값을 썼는지 확인하려면 `stream(입력, stream_mode="updates")`가 내놓는 값을 순서대로 읽습니다. 같은 그래프에 긴 문장을 넣어, 그림의 두 작업이 아래 출력의 어느 줄에 나타나는지 확인하세요.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    count_chars["count_chars · count에 15 기록"]
    decide_label["decide_label · label에 줄여 주세요 기록"]
    finish["END · 실행 마치기"]
    start --> count_chars
    count_chars --> decide_label
    decide_label --> finish
```

In [4]:
for update in notice_graph.stream({"text": "창문과 출입문을 닫아 주세요"}, stream_mode="updates"):
    print(update)

{'count_chars': {'count': 15}}
{'decide_label': {'label': '줄여 주세요'}}


첫 줄의 바깥 키는 `count_chars`이고 안쪽에는 이 노드가 기록한 `count`가 있습니다. 둘째 줄의 `decide_label`은 그 수를 보고 “줄여 주세요”를 기록합니다. `stream`도 새 실행을 시작합니다. 모델을 부르는 그래프에서 같은 입력을 `invoke`와 `stream`으로 각각 실행하면 모델 호출도 다시 일어날 수 있습니다.

## 앞 단계가 아직 쓰지 않은 칸을 읽으면

작업 순서가 뒤집힌 상황을 재현합니다. 시작 상태에는 text만 있으니 둘째 노드가 무엇을 찾지 못할까요? 오류를 일부러 만든 뒤 정확한 예외 이름을 확인하면, 나중에 큰 그래프에서도 모델과 무관한 상태 연결 오류를 따로 찾을 수 있습니다.

In [5]:
try:
    decide_label({"text": "오늘 정상 운영"})
except KeyError as error:
    # 일부러 빠뜨린 상태 필드 이름을 학생이 확인합니다.
    print("준비되지 않은 칸", error.args[0])

준비되지 않은 칸 count


여기서 count 기본값을 0으로 채우면 프로그램은 지나가지만 긴 문장도 표시 가능으로 판정합니다. 입력에 count를 잘못 넣은 문제인지, 연결 순서가 뒤집힌 문제인지 나누기 위해 첫 노드가 실제로 반환한 작은 딕셔너리를 봅니다.

## 실습 · 문장 길이 9·10·11에서 판정 확인하기

같은 그래프에 길이만 다른 세 문장을 넣습니다. 그림에서 10글자를 판정하는 노드를 찾고, 아래 셀에서 세 번 실행해 보세요. 각 실행의 최종 상태에서 `count`와 `label`을 출력합니다.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    count_chars["count_chars · 글자 수 세기"]
    decide_label["decide_label · 10자 이하 표시 가능, 초과 시 줄여 주세요"]
    finish["END · 실행 마치기"]
    start --> count_chars
    count_chars --> decide_label
    decide_label --> finish
```

In [6]:
lengths = [9, 10, 11]
# 각 길이만큼 "가"를 반복해 notice_graph.invoke의 text에 넣고 count와 label을 출력하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>for text in [&quot;가&quot; * 9, &quot;가&quot; * 10, &quot;가&quot; * 11]:
    result = notice_graph.invoke({&quot;text&quot;: text})
    print(result[&quot;count&quot;], result[&quot;label&quot;])</code></pre><p>실행 결과</p><pre>9 표시 가능
10 표시 가능
11 줄여 주세요</pre><p>9자와 10자는 표시 가능, 11자는 줄여 달라는 결과입니다. 조건을 &lt; 10으로 바꾸면 딱 10자인 입력에서만 결과가 바뀝니다.</p></details>

## 실습 · 남은 칸을 계산하는 노드 덧붙이기

`remaining_space`는 `count`를 읽고 10칸 가운데 남은 수를 `remaining`에 씁니다. 11글자일 때도 남은 자리는 0칸입니다. `BoardState`에 새 필드를 선언하고, 아래 그림처럼 기존 두 노드 뒤에 세 번째 함수를 연결하세요. `add_sequence([함수1, 함수2, 함수3])`는 함수 이름으로 노드를 등록하고 목록 순서대로 이웃한 두 노드를 잇습니다. 시작과 끝을 연결하는 간선은 별도로 적습니다.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    count_chars["count_chars · 글자 수 세기"]
    decide_label["decide_label · 표시 문구 정하기"]
    remaining_space["remaining_space · 남은 칸 수 계산하기"]
    finish["END · 실행 마치기"]
    start --> count_chars
    count_chars --> decide_label
    decide_label --> remaining_space
    remaining_space --> finish
```

In [7]:
test_text = "가" * 11
# remaining을 선언하고 남은 칸 수를 계산하는 함수를 만드세요.
# 세 노드를 연결하고 test_text를 넣어 최종 상태를 출력하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>class BoardState(NoticeState, total=False):
    remaining: int

def remaining_space(state):
    return {&quot;remaining&quot;: max(0, 10 - state[&quot;count&quot;])}

board = StateGraph(BoardState)
board.add_sequence([count_chars, decide_label, remaining_space])
board.add_edge(START, &quot;count_chars&quot;)
board.add_edge(&quot;remaining_space&quot;, END)
compiled_board = board.compile()
print(compiled_board.invoke({&quot;text&quot;: &quot;가&quot; * 11}))</code></pre><p>실행 결과</p><pre>{&#x27;text&#x27;: &#x27;가가가가가가가가가가가&#x27;, &#x27;count&#x27;: 11, &#x27;label&#x27;: &#x27;줄여 주세요&#x27;, &#x27;remaining&#x27;: 0}</pre><p>새 칸을 선언하고 글자 수를 읽는 마지막 작업을 연결했습니다. 11글자일 때 count는 11, remaining은 0입니다. add_sequence는 나열한 함수 사이의 간선을 순서대로 연결합니다.</p></details>

## 실습 · 빈 문장을 판정하는 함수로 노드 교체하기

앞의 그래프에 빈 문자열을 넣으면 길이가 0이어서 “표시 가능”으로 끝납니다. 이번에는 빈 문자열이나 공백만 든 문장을 받으면 “원고를 입력하세요”라고 알려 주도록 고칩니다. 글자 수를 세는 첫 노드는 그대로 씁니다.

둘째 노드를 `review_notice`로 교체하세요. 이 함수 안에서 `text.strip()`으로 내용이 있는지 확인하고, 글자가 있으면 앞에서 정의한 `decide_label`을 호출합니다. 그림에서 `review_notice`가 노드 한 개인 이유는 두 판단이 같은 함수 안에서 일어나기 때문입니다. 공백 세 칸을 넣으면 `count`는 3으로 남아야 합니다.

```mermaid
flowchart TD
    start["START · 입력 받기"]
    count_chars["count_chars · 공백까지 글자 수 세기"]
    review_notice["review_notice · 빈 글 확인 후 표시 문구 정하기"]
    finish["END · 실행 마치기"]
    start --> count_chars
    count_chars --> review_notice
    review_notice --> finish
```

In [8]:
drafts = ["", "   ", "동문 개방"]
# review_notice를 작성한 뒤 count_chars와 이어 보세요.
# 각 입력에서 원래 글자 수와 새 표시 문구를 출력하세요.

<details><summary>정답 코드·실행 결과·풀이</summary>
<pre><code>def review_notice(state):
    if not state[&quot;text&quot;].strip():
        return {&quot;label&quot;: &quot;원고를 입력하세요&quot;}
    return decide_label(state)

plan = StateGraph(NoticeState)
plan.add_sequence([count_chars, review_notice])
plan.add_edge(START, &quot;count_chars&quot;)
plan.add_edge(&quot;review_notice&quot;, END)
review_graph = plan.compile()

for text in [&quot;&quot;, &quot;   &quot;, &quot;동문 개방&quot;]:
    result = review_graph.invoke({&quot;text&quot;: text})
    print(result[&quot;count&quot;], result[&quot;label&quot;])</code></pre>
<p>실행 결과</p>
<pre>0 원고를 입력하세요
3 원고를 입력하세요
5 표시 가능</pre>
<p>공백을 제거한 값은 내용 검사에만 썼습니다. 첫 노드가 원문에서 센 길이 3은 상태에 남고, 둘째 노드가 표시 문구만 갱신합니다.</p>
</details>

다음 장에서는 긴 원고만 수정하는 경로를 만듭니다. 지금 그린 화살표 가운데 어느 연결이 입력에 따라 달라져야 할지 생각해 보세요.

[LangGraph 공식 문서 · Graph API의 상태, 노드, 간선](https://docs.langchain.com/oss/python/langgraph/graph-api)